# Sprint 57 — Wall polygon U-Net 학습 (Colab)

1. 위 셀 실행 순서대로 진행
2. `wall_polygon_dl_colab.zip`을 Files 패널에 드래그 업로드
3. 학습 끝나면 `runs/best.pt` 다운로드 → 로컬 `infer.py`에 사용

**예상 시간**: T4 GPU 기준 1k sample × 30 epoch ≈ 10–15분

## 0. Runtime → GPU 설정 확인

In [ ]:
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('GPU 미할당. Runtime → Change runtime type → T4 GPU 선택 후 재실행')

## 1. 패키지 설치

In [ ]:
!pip install -q segmentation-models-pytorch opencv-python-headless shapely

## 2. 코드 + 데이터 zip 업로드

왼쪽 Files 패널 (📁 아이콘) → 위쪽 업로드 버튼 → `wall_polygon_dl_colab.zip` 선택.

업로드 끝나면 아래 셀 실행.

In [ ]:
import os, zipfile
ZIP = 'wall_polygon_dl_colab.zip'
assert os.path.exists(ZIP), f'{ZIP} 가 /content/ 에 없습니다. 위 패널에서 업로드 후 다시 실행하세요.'
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/work')
print('extracted to /content/work')
print(sorted(os.listdir('/content/work')))

## 3. 학습

**기본**: resnet34 + 512×512 + batch 8 + 30 epoch (T4 ~12분).

필요 시 `EPOCHS`, `BATCH`, `RESIZE`, `ENCODER` 조절.

In [ ]:
import os, sys
WORK = '/content/work'
sys.path.insert(0, WORK)
os.chdir(WORK)

EPOCHS = 30
BATCH = 8
RESIZE = 512
ENCODER = 'resnet34'
TRAIN = 900   # cache 의 0..899
VAL   = 100   # cache 의 900..999
CACHE = '/content/work/cache_v1'
OUT   = '/content/work/runs/poc_v2'

!python train.py \
  --train {TRAIN} --val {VAL} \
  --epochs {EPOCHS} --batch {BATCH} \
  --lr 1e-3 --resize-to {RESIZE} \
  --encoder {ENCODER} \
  --device cuda --num-workers 2 \
  --cache-dir {CACHE} \
  --out {OUT}

## 4. best.pt 다운로드

왼쪽 Files 패널에서 `work/runs/poc_v2/best.pt` 우클릭 → Download.

또는 아래 셀로 자동 다운로드.

In [ ]:
from google.colab import files
files.download('/content/work/runs/poc_v2/best.pt')
files.download('/content/work/runs/poc_v2/history.json')
files.download('/content/work/runs/poc_v2/config.json')

## (옵션) Google Drive에 저장

Colab 12시간 후 disconnect되어도 weight 보존.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
DEST = '/content/drive/MyDrive/wall_polygon_dl/poc_v2'
os.makedirs(DEST, exist_ok=True)
for f in ['best.pt', 'history.json', 'config.json']:
    src = f'/content/work/runs/poc_v2/{f}'
    if os.path.exists(src):
        shutil.copy(src, DEST)
print('saved to', DEST)